# Plan your trip with Kayak : Data Engineering Pipeline

Contexte Stratégique
L'équipe marketing de Kayak souhaite lancer un outil de recommandation de destinations en France basé sur des données réelles (météo et hôtellerie), afin de rassurer 70% des utilisateurs hésitants.

Objectifs du Pipeline (ETL)

Extraction : Collecter les données GPS et météorologiques via API (OpenWeather) et les données hôtelières via Web Scraping (Booking.com), dans le strict respect du RGPD (aucune donnée personnelle).

Stockage (Data Lake) : Sécuriser la donnée brute sur le Cloud (Amazon S3) pour garantir la scalabilité (Big Data) et optimiser les coûts.

Transformation & Chargement : Nettoyer les données (Pandas) et les charger dans un Data Warehouse (Amazon RDS - PostgreSQL) pour les rendre requêtables par les équipes métiers.

In [1]:
# Librairies standards pour la gestion du système et du temps
import os
import time

# Librairies pour la collecte de données (Extraction)
import requests

# Librairies pour la manipulation de données (Transformation)
import pandas as pd

# Librairies pour la visualisation (Livraison)
import plotly.express as px

In [2]:
# Liste des 35 villes ciblées par le cahier des charges de la direction marketing
cities = [
    "Mont Saint Michel", "St Malo", "Bayeux", "Le Havre", "Rouen", "Paris", 
    "Amiens", "Lille", "Strasbourg", "Chateau du Haut Koenigsbourg", "Colmar", 
    "Eguisheim", "Besancon", "Dijon", "Annecy", "Grenoble", "Lyon", 
    "Gorges du Verdon", "Bormes les Mimosas", "Cassis", "Marseille", 
    "Aix en Provence", "Avignon", "Uzes", "Nimes", "Aigues Mortes", 
    "Saintes Maries de la mer", "Collioure", "Carcassonne", "Ariege", 
    "Toulouse", "Montauban", "Biarritz", "Bayonne", "La Rochelle"
]

# Étape 1 : Enrichissement Géographique (API Nominatim)
Pour interroger l'API météorologique, nous devons d'abord convertir les noms des villes en coordonnées GPS exactes (Latitude / Longitude).

In [3]:
def get_gps_coordinates(city_list):
    """
    Interroge l'API publique Nominatim pour récupérer la latitude et la longitude.
    Intègre une pause (time.sleep) pour respecter les conditions d'utilisation du serveur.
    """
    gps_data = []
    
    for city in city_list:
        # Configuration de l'URL et des paramètres requis par l'API
        url = "https://nominatim.openstreetmap.org/search"
        params = {
            "q": city + ", France", # Précision du pays pour éviter les homonymes mondiaux
            "format": "json",
            "limit": 1
        }
        # Déclaration de notre identité pour ne pas être bloqué (Bonne pratique)
        headers = {"User-Agent": "Kayak_Data_Pipeline_Project/1.0"}
        
        try:
            response = requests.get(url, params=params, headers=headers)
            
            # Si le serveur répond correctement (Code 200 = Succès)
            if response.status_code == 200 and len(response.json()) > 0:
                data = response.json()[0]
                gps_data.append({
                    "city": city,
                    "lat": float(data["lat"]),
                    "lon": float(data["lon"])
                })
            else:
                print(f"Coordonnées introuvables pour : {city}")
                
        except Exception as e:
            print(f"Erreur technique pour {city} : {e}")
            
        # Respect de la limite de l'API gratuite (1 requête par seconde)
        time.sleep(1)
        
    return pd.DataFrame(gps_data)

# Exécution de la fonction et affichage des 5 premiers résultats
df_cities = get_gps_coordinates(cities)
display(df_cities.head())

,city,lat,lon
0,Mont Saint Michel,48.635954,-1.511460
1,St Malo,48.649518,-2.026041
2,Bayeux,49.276462,-0.702474
3,Le Havre,49.493898,0.107973
4,Rouen,49.440459,1.093966


In [4]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # va chercher le fichier .env tout seul
API_KEY = os.getenv("OPENWEATHER_API_KEY")
print("Clé chargée :", API_KEY is not None)  # doit afficher True

def get_weather_data(df):
    """
    Parcourt le DataFrame des villes et interroge l'API OpenWeather 
    pour récupérer la température moyenne et la probabilité de pluie.
    """
    weather_data = []
    
    # itterrows() permet de lire le tableau ligne par ligne
    for index, row in df.iterrows():
        url = "https://api.openweathermap.org/data/2.5/forecast"
        params = {
            "lat": row["lat"],
            "lon": row["lon"],
            "units": "metric", # Pour avoir des degrés Celsius
            "appid": API_KEY
        }
        
        try:
            response = requests.get(url, params=params)
            
            if response.status_code == 200:
                data = response.json()
                
                # Extraction des données sur les prévisions (liste 'list' dans le JSON)
                forecasts = data["list"]
                
                # Calcul de la température moyenne sur les prévisions récupérées
                avg_temp = sum([f["main"]["temp"] for f in forecasts]) / len(forecasts)
                # Calcul de la probabilité de pluie moyenne (pop = Probability of Precipitation)
                avg_rain_prob = sum([f["pop"] for f in forecasts]) / len(forecasts)
                
                weather_data.append({
                    "city": row["city"],
                    "avg_temp": round(avg_temp, 2),
                    "rain_prob": round(avg_rain_prob, 2)
                })
            else:
                print(f"Erreur API Météo pour {row['city']} (Code: {response.status_code})")
                
        except Exception as e:
            print(f"Erreur technique pour {row['city']} : {e}")
            
    return pd.DataFrame(weather_data)

# Exécution de la fonction
df_weather = get_weather_data(df_cities)

# Fusion (Merge) des coordonnées GPS avec les données météo
df_final_weather = pd.merge(df_cities, df_weather, on="city")
display(df_final_weather.head())

Clé chargée : True


,city,lat,lon,avg_temp,rain_prob
0,Mont Saint Michel,48.635954,-1.511460,15.49,0.17
1,St Malo,48.649518,-2.026041,16.11,0.23
2,Bayeux,49.276462,-0.702474,14.71,0.15
3,Le Havre,49.493898,0.107973,16.00,0.35
4,Rouen,49.440459,1.093966,14.10,0.17


# Étape 3 : Classement des Top 5 Destinations & Sauvegarde (Data Lake)
Objectif : Créer un score météo (KPI) pour classer automatiquement les villes et isoler le Top 5. Sauvegarder ensuite ce résultat brut sous forme de fichier CSV, qui servira de point de départ pour notre Data Lake Amazon S3.

In [5]:
# Création d'un algorithme de score basique : on valorise la chaleur et on pénalise la pluie
# Note : pop (rain_prob) est un pourcentage entre 0 et 1. On le multiplie par 10 pour que la pénalité soit visible.
df_final_weather["weather_score"] = df_final_weather["avg_temp"] - (df_final_weather["rain_prob"] * 10)

# Classement des villes de la meilleure à la pire selon notre score
df_sorted = df_final_weather.sort_values(by="weather_score", ascending=False)

# Isolement du Top 5 demandé par le marketing
top_5_cities = df_sorted.head(5)
display(top_5_cities)

# Sauvegarde physique des données (Création du fichier pour le Data Lake)
# On sauvegarde le dataset complet
df_final_weather.to_csv("kayak_weather_dataset.csv", index=False)
# On sauvegarde aussi la liste stricte du Top 5 pour nos futurs robots de scraping
top_5_cities.to_csv("top_5_destinations.csv", index=False)

print("Étape validée : Fichiers CSV générés avec succès !")

,city,lat,lon,avg_temp,rain_prob,weather_score
33,Bayonne,43.494514,-1.473666,18.04,0.07,17.34
27,Collioure,42.525050,3.083155,17.28,0.00,17.28
32,Biarritz,43.483252,-1.559278,17.97,0.07,17.27
18,Bormes les Mimosas,43.150697,6.341928,16.84,0.04,16.44
26,Saintes Maries de la mer,43.451592,4.427720,15.97,0.02,15.77


Étape validée : Fichiers CSV générés avec succès !


# Étape 4 : Intégration et Fiabilisation des Données Hôtelières
Le robot d'extraction (Scrapy/Selenium) a fonctionné en arrière-plan pour collecter les informations de Booking.com sur notre Top 5. Nous importons ici sa collecte brute pour la nettoyer (suppression du texte inutile, conversion en nombres) et la fusionner avec nos données météorologiques.

In [6]:
import json

# 1. Chargement du fichier brut généré par le robot de scraping
with open("hotels_raw.json", "r", encoding="utf-8") as f:
    hotels_data = json.load(f)
    
df_hotels = pd.DataFrame(hotels_data)

# 2. Nettoyage de la colonne 'rating' (Transformation ETL)
# Le robot ramène du texte brut comme "Avec une note de 8,4" ou "8,4"
# On arrache le texte pour ne garder que les chiffres, on remplace la virgule par un point, et on convertit en décimal (float)
df_hotels['hotel_score'] = df_hotels['rating'].str.extract(r'(\d+,\d+|\d+)')[0].str.replace(',', '.').astype(float)

# 3. Fusion des Hôtels avec le Top 5 Météo (La consolidation)
df_final = pd.merge(top_5_cities, df_hotels, on="city", how="inner")

# On affiche les colonnes essentielles pour vérifier le nettoyage
display(df_final[["city", "name", "hotel_score", "avg_temp", "weather_score"]].head())

,city,name,hotel_score,avg_temp,weather_score
0,Bayonne,ibis budget Bayonne,7.7,18.04,17.34
1,Bayonne,Hotel Cote Basque,8.1,18.04,17.34
2,Bayonne,Hôtel Les Genêts Bayonne,7.5,18.04,17.34
3,Bayonne,Premiere Classe Bayonne,7.1,18.04,17.34
4,Bayonne,Adonis Hotel Bayonne,7.6,18.04,17.34


# Étape 5 : Déploiement Cloud (Amazon S3 & RDS)
Processus ETL (Load) : Le dataset final est d'abord sauvegardé en local. Dans un environnement de production, il est ensuite expédié vers notre Data Lake (Amazon S3) pour l'archivage brut, puis injecté dans notre Data Warehouse (Amazon RDS - PostgreSQL) pour être interrogeable en SQL par les équipes marketing.

In [10]:
import os

import boto3
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import create_engine

load_dotenv(find_dotenv())

# 1. Sauvegarde locale du fichier final (la preuve physique)
df_final.to_csv("kayak_final_dataset.csv", index=False)
print("✅ Fichier local enregistré :", len(df_final), "hôtels")

# 2. Envoi vers AWS : passer à True seulement le jour où on envoie vraiment
ENVOYER_SUR_AWS = False

if ENVOYER_SUR_AWS:
    # Étape S3 : le « garage » où l'on dépose le fichier tel quel
    session = boto3.Session(
        aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
        aws_secret_access_key=os.getenv("AWS_SECRET_KEY"),
        region_name="eu-west-3",
    )
    s3 = session.resource("s3")
    s3.Bucket("kayak-datalake-jedha-carelle-2026").upload_file(
        "kayak_final_dataset.csv", "kayak_final_dataset.csv"
    )
    print("✅ Fichier envoyé sur S3")

    # Étape RDS : la base de données où l'on pose des questions en SQL
    mot_de_passe = os.getenv("RDS_PASSWORD")
    adresse = os.getenv("RDS_ENDPOINT")
    engine = create_engine(
        f"postgresql://postgres:{mot_de_passe}@{adresse}:5432/postgres"
    )
    df_final.to_sql("destinations_hotels", con=engine, if_exists="replace", index=False)
    print("✅ Table 'destinations_hotels' créée dans RDS")
else:
    print("⏸️ Envoi AWS non lancé (ENVOYER_SUR_AWS = False).")

✅ Fichier local enregistré : 100 hôtels
⏸️ Envoi AWS non lancé (ENVOYER_SUR_AWS = False).


In [8]:
import plotly.express as px
import requests
import time

# ==========================================
# CARTE 1 : LES 5 MEILLEURES DESTINATIONS
# ==========================================
fig_destinations = px.scatter_mapbox(
    top_5_cities, 
    lat="lat", 
    lon="lon", 
    hover_name="city",
    hover_data={"avg_temp": True, "rain_prob": True, "lat": False, "lon": False},
    color="avg_temp",
    color_continuous_scale="Bluered",
    size="avg_temp",
    zoom=5, 
    center={"lat": 46.6033, "lon": 2.2137},
    mapbox_style="open-street-map",
    title="Top 5 des destinations les plus ensoleillées"
)
fig_destinations.update_traces(marker=dict(size=22, opacity=0.9))
fig_destinations.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig_destinations.show()

/var/folders/d2/2h9xzbcd25x76wt_vzczswgm0000gn/T/ipykernel_6769/211275621.py:8: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_destinations = px.scatter_mapbox(


In [9]:
import pandas as pd

df_frais = pd.read_json('csv/hotels_raw.json')

print("Colonnes exactes du fichier :", df_frais.columns.tolist())
display(df_frais.head(2))

FileNotFoundError: File csv/hotels_raw.json does not exist

In [ ]:
import pandas as pd
import plotly.express as px

# On prend les 20 hôtels les mieux notés dans NOS données réelles (df_final)
top_20_hotels = df_final.sort_values("hotel_score", ascending=False).head(20)
top_20_hotels = top_20_hotels.reset_index(drop=True)

# Les hôtels d'une même ville ont les mêmes coordonnées (celles de la ville).
# On les décale très légèrement pour les voir séparément sur la carte.
top_20_hotels["lat"] = top_20_hotels["lat"] + (top_20_hotels.index % 5) * 0.006
top_20_hotels["lon"] = top_20_hotels["lon"] + (top_20_hotels.index // 5) * 0.006

fig_hotels = px.scatter_mapbox(
    top_20_hotels,
    lat="lat",
    lon="lon",
    hover_name="name",
    hover_data={"city": True, "hotel_score": True, "lat": False, "lon": False},
    color="city",
    zoom=6,
    center={"lat": 43.6, "lon": 3.8},
    mapbox_style="open-street-map",
    title="Top 20 : meilleurs hôtels (notes Booking)",
)
fig_hotels.update_traces(marker=dict(size=14, opacity=0.9))
fig_hotels.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0})
fig_hotels.show()

/var/folders/d2/2h9xzbcd25x76wt_vzczswgm0000gn/T/ipykernel_4582/1160916952.py:13: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig_hotels = px.scatter_mapbox(
